In [ ]:
from pathlib import Path
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, balanced_accuracy_score

In [ ]:
data_path = Path.cwd().parent / "artifacts" / "isic2019_verified_split.csv" #needs to be isic2019_binary_split.csv, but neither notebook 1 nor 2 currently make that???

df = pd.read_csv(data_path)
print(df.head())
print(df.columns)
print(df["split"].value_counts())
print(df.groupby(["split", "label"]).size().unstack(fill_value=0))

In [ ]:
train_df = df[df["split"] == "train"].copy()
val_df   = df[df["split"] == "validation"].copy()
test_df  = df[df["split"] == "test"].copy()

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

# Create a consistent mapping of class names to integer IDs
class_names = sorted(df["label"].unique().tolist())
class_to_id = {name: i for i, name in enumerate(class_names)}

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(
        image, channels=3, expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def make_dataset(frame, training=False):
    paths = frame["path"].astype(str).values
    labels = frame["label"].map(class_to_id).values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        ds = ds.shuffle(
            buffer_size=len(frame),
            reshuffle_each_iteration=True
        )

    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

In [ ]:
y_train = train_df["label"].map(class_to_id).to_numpy()
classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = {
    int(cls): float(weight)
    for cls, weight in zip(classes, weights)
}

print(class_weights)
class_to_num = class_to_id
num_to_class = {v: k for k, v in class_to_num.items()}

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(128, 128, 3)),

    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(256, 3, padding="same", activation="relu"),
    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.35),
    tf.keras.layers.Dense(len(class_names), activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        Path.cwd().parent / "artifacts" / "cnn_conventional_aug.keras",
        monitor="val_loss",
        save_best_only=True,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
    ),
]

In [ ]:
translation_factors = [0.00, 0.01, 0.025, 0.05, 0.10, 0.25]

results = []

for factor in translation_factors:
    print(f"\nTesting translation factor {factor}")

    augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomTranslation(
            height_factor=factor,
            width_factor=factor,
            fill_mode="reflect", seed=104
        )
    ], name="conventional_augmentation")

    aug_train_ds = train_ds.map(
        lambda x, y: (
            tf.clip_by_value(augmentation(x, training=True), 0., 1.), y
        ),
        num_parallel_calls=1,
    ).prefetch(tf.data.AUTOTUNE)

In [ ]:
history = model.fit(
    aug_train_ds,
    validation_data=val_ds,
    epochs=25,
    verbose=2,
    class_weight=class_weights,
    callbacks=callbacks,
)

In [ ]:
print("Best validation loss:", min(history.history["val_loss"]))
print("Epochs trained:", len(history.history["loss"]))

# Evaluate the best saved baseline on validation only (leave test untouched)
best_model =  Path.cwd().parent / "artifacts" / "cnn_conventional_aug.keras"
val_part = df.loc[df['split'] == 'val'].copy()
y_true = val_part['label'].map(class_to_num).to_numpy(dtype=int)
probabilities = best_model.predict(val_ds, verbose=0)
y_pred = probabilities.argmax(axis=1)
assert len(y_true) == len(y_pred) == 355

print('Validation accuracy:', round(accuracy_score(y_true, y_pred), 4))
print('Validation macro F1:', round(f1_score(y_true, y_pred, labels=np.arange(len(class_names)), average='macro', zero_division=0), 4))
print('Validation balanced accuracy:', round(balanced_accuracy_score(y_true, y_pred), 4))
print('\nPer-class report:')
print(classification_report(y_true, y_pred, labels=np.arange(len(class_names)), target_names=class_names, zero_division=0))
cm = pd.DataFrame(confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names))), index=class_names, columns=class_names)
print('Confusion matrix (rows=true, columns=predicted):')
print(cm.to_string())
cm.to_csv('/kaggle/working/conventional_aug_validation_confusion_matrix.csv')
val_part['predicted_label'] = [class_names[i] for i in y_pred]
val_part['confidence'] = probabilities.max(axis=1)
val_part[['image_id', 'label', 'predicted_label', 'confidence']].to_csv('/kaggle/working/conventional_aug_validation_predictions.csv', index=False)
print('Saved validation metrics files in /kaggle/working')
import json
from sklearn.metrics import classification_report
pd.DataFrame(history.history).assign(epoch=lambda d: np.arange(1, len(d)+1)).to_csv('/kaggle/working/conventional_aug_history.csv', index=False)
metrics = {
    'experiment': 'conventional_augmentation', 'seed': 42,
    'epochs_trained': len(history.history['loss']),
    'best_epoch': int(np.argmin(history.history['val_loss']) + 1),
    'best_validation_loss': float(min(history.history['val_loss'])),
    'validation_accuracy': float(accuracy_score(y_true, y_pred)),
    'validation_macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
    'validation_balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
    'augmentation': {'flip': 'horizontal_and_vertical', 'rotation_turn_fraction': 0.05, 'zoom_factor': 0.10, 'fill_mode': 'reflect'},
    'tensorflow_version': tf.__version__,
    'classification_report': classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0),
    'test_evaluated': False,
}
Path('/kaggle/working/conventional_aug_metrics.json').write_text(json.dumps(metrics, indent=2))
print('EXPERIMENT_COMPLETE', json.dumps(metrics))

